# Experiment 3: How many max features?

TF-IDF trigrams (the winner of experiment 2) with 1,000 to 10,000 features.

In [1]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000'))
EXPERIMENT = 'Exp 3 - TfIdf Trigram max_features'
mlflow.set_experiment(EXPERIMENT)
BATCH = datetime.now().strftime('%Y%m%d-%H%M%S')

df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)
df.shape

2026/09/11 11:03:49 INFO mlflow.tracking.fluent: Experiment with name 'Exp 3 - TfIdf Trigram max_features' does not exist. Creating a new experiment.


(36662, 2)

In [2]:
def log_evaluation(y_true, y_pred, title):
    """Log accuracy, per-class metrics and a confusion matrix to the active MLflow run."""
    mlflow.log_metric('accuracy', accuracy_score(y_true, y_pred))
    for label, metrics in classification_report(y_true, y_pred, output_dict=True).items():
        if isinstance(metrics, dict):
            mlflow.log_metrics({f'{label}_{name}': value for name, value in metrics.items()})

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set(xlabel='Predicted', ylabel='Actual', title=f'Confusion Matrix: {title}')
    mlflow.log_figure(fig, 'confusion_matrix.png')
    plt.close(fig)


def compare_runs():
    """Table of this batch's runs; the video reads the same numbers off MLflow's parallel-coordinates plot."""
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.batch = '{BATCH}'")
    columns = {
        'tags.mlflow.runName': 'run',
        'metrics.accuracy': 'accuracy',
        'metrics.-1_precision': 'neg_precision',
        'metrics.-1_recall': 'neg_recall',
        'metrics.1_precision': 'pos_precision',
        'metrics.1_recall': 'pos_recall',
    }
    return runs[list(columns)].rename(columns=columns).sort_values('accuracy', ascending=False).round(4)

In [3]:
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    with mlflow.start_run(run_name=f'TFIDF_Trigrams_max_features_{max_features}'):
        mlflow.set_tags({'experiment_type': 'feature_engineering', 'model_type': 'RandomForestClassifier', 'batch': BATCH})
        mlflow.log_params({
            'vectorizer_type': 'TF-IDF',
            'ngram_range': ngram_range,
            'vectorizer_max_features': max_features,
            'n_estimators': 200,
            'max_depth': 15,
        })

        model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        log_evaluation(y_test, model.predict(X_test), f'TF-IDF trigrams, max_features={max_features}')


for max_features in range(1000, 10001, 1000):
    run_experiment_tfidf_max_features(max_features)

2026/09/11 11:03:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run TFIDF_Trigrams_max_features_1000 at: http://127.0.0.1:5000/#/experiments/3/runs/7d39d004aac04549abad8bc1b474a4c4.


2026/09/11 11:03:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


2026/09/11 11:04:05 INFO mlflow.tracking._tracking_service.client: 🏃 View run TFIDF_Trigrams_max_features_2000 at: http://127.0.0.1:5000/#/experiments/3/runs/4b498beead5c400b9447a87d0b11a843.


2026/09/11 11:04:05 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


2026/09/11 11:04:11 INFO mlflow.tracking._tracking_service.client: 🏃 View run TFIDF_Trigrams_max_features_3000 at: http://127.0.0.1:5000/#/experiments/3/runs/d80e44fbaf214ffba41f8c9fa4398414.


...

2026/09/11 11:04:44 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


2026/09/11 11:04:50 INFO mlflow.tracking._tracking_service.client: 🏃 View run TFIDF_Trigrams_max_features_9000 at: http://127.0.0.1:5000/#/experiments/3/runs/62a96d62709449fba3836796fbb0e31d.


2026/09/11 11:04:50 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


2026/09/11 11:04:56 INFO mlflow.tracking._tracking_service.client: 🏃 View run TFIDF_Trigrams_max_features_10000 at: http://127.0.0.1:5000/#/experiments/3/runs/fd2999aa9e2748678b5671f5623acc3c.


2026/09/11 11:04:56 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


In [4]:
compare_runs()

,run,accuracy,neg_precision,neg_recall,pos_precision,pos_recall
9,TFIDF_Trigrams_max_features_1000,0.6618,0.8775,0.1345,0.6867,0.7521
8,TFIDF_Trigrams_max_features_2000,0.6547,0.9021,0.0782,0.6572,0.7879
3,TFIDF_Trigrams_max_features_7000,0.6520,0.9714,0.0206,0.6335,0.8386
7,TFIDF_Trigrams_max_features_3000,0.6516,0.9394,0.0376,0.6515,0.8066
6,TFIDF_Trigrams_max_features_4000,0.6498,0.9767,0.0255,0.6439,0.8174
4,TFIDF_Trigrams_max_features_6000,0.6482,0.9615,0.0152,0.6318,0.8345
5,TFIDF_Trigrams_max_features_5000,0.6473,0.9714,0.0206,0.6375,0.8209
1,TFIDF_Trigrams_max_features_9000,0.6469,1.0000,0.0133,0.6198,0.8580
0,TFIDF_Trigrams_max_features_10000,0.6460,1.0000,0.0079,0.6209,0.8580
2,TFIDF_Trigrams_max_features_8000,0.6446,0.9565,0.0133,0.6160,0.8561


The video settles on **1,000 features**: fewer features gave higher negative-class recall and accuracy.
All later experiments use TF-IDF (1,3) with 1,000 features. The original notebooks quietly switched
back to 10,000 in experiments 4, 5 and 7; that inconsistency is fixed here.